# Laboratório — Lógica fuzzy: ventilação de um painel elétrico

**Inteligência Artificial aplicada à Engenharia Elétrica · Projeto 2 · 25/11/2026**

Complementa o [simulador](https://furriel.github.io/IA/fuzzy/): aqui o controlador é escrito do zero em Python (sem biblioteca fuzzy), com as mesmas funções de pertinência, regras e defuzzificação da página, e analisado com gráficos e testes.

**Execução:** baixe o `.ipynb` e use *Arquivo → Fazer upload de notebook* no Colab, ou abra no Jupyter. Requer apenas NumPy e Matplotlib.

> Faixas didáticas, não normativas. O controlador é estático (sem inércia térmica) e não deve acionar equipamento real.

## Antes de começar

### Objetivo
Reconstruir o controlador fuzzy da aula 3 em Python, reproduzir V* ≈ 73,6 % para 45 °C e 70 % e encontrar, na superfície de controle, o vale em que mais carga reduz a ventilação.

### Contexto do problema
Um painel elétrico precisa de ventilação proporcional à temperatura T e ao carregamento L. Nove regras SE/ENTÃO do tipo Mamdani, com mínimo, máximo e centroide, convertem T e L em um comando V*.

### Pré-requisitos
Nenhum conhecimento prévio de Python. Faça antes, no papel, o cálculo da etapa *Calcular* da aula 3 (pertinências e forças das regras para 45 °C / 70 %).

### Como executar
- **Google Colab:** abra o notebook pelo link da aula, conecte o ambiente e use *Ambiente de execução → Executar tudo*. Para guardar suas alterações, use *Arquivo → Salvar uma cópia no Drive*.
- **Kaggle:** em *kaggle.com/code*, crie um notebook, use *File → Import Notebook* e envie o arquivo `.ipynb`; depois *Run → Run All*.
- **Jupyter local:** requer Python 3 com NumPy e Matplotlib.
- Execute as células **na ordem**; uma célula depende das anteriores.
- Passo a passo ilustrado, com solução de problemas: https://furriel.github.io/IA/apoio/executar-python.html

### Dados e parâmetros
Não há dados externos. Parâmetros que você pode alterar, **um de cada vez**: os argumentos de `inferir(T, L)`, os números dos conjuntos em `VARS` e a lista `REGRAS`. Depois de qualquer mudança, execute de novo desde a célula que define `VARS`.

## 1. Problema

A temperatura interna $T$ de um painel (20–70 °C) depende do carregamento $L$ dos circuitos (0–100 %). Um termostato liga/desliga trata 44,9 °C e 45,1 °C como situações opostas. O objetivo é um comando de ventilação $V$ (0–100 %) que varie gradualmente segundo regras de especialista.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

def trapezio(x, a, b, c, d):
    # a <= b <= c <= d; triângulo quando b == c; ombro quando a == b ou c == d
    if x < a or x > d: return 0.0
    if b <= x <= c: return 1.0
    if x < b: return 1.0 if b == a else (x - a) / (b - a)
    return 1.0 if d == c else (d - x) / (d - c)

VARS = {
  "T": {"nome": "Temperatura interna", "unidade": "°C", "dom": (20, 70),
        "conj": {"baixa": (20, 20, 30, 40), "média": (30, 45, 45, 60), "alta": (50, 60, 70, 70)}},
  "L": {"nome": "Carregamento", "unidade": "%", "dom": (0, 100),
        "conj": {"leve": (0, 0, 20, 45), "nominal": (25, 50, 50, 75), "elevado": (55, 80, 100, 100)}},
  "V": {"nome": "Ventilação", "unidade": "%", "dom": (0, 100),
        "conj": {"lenta": (0, 0, 15, 35), "moderada": (25, 50, 50, 75), "rápida": (65, 85, 100, 100)}}}
REGRAS = [("baixa", "leve", "lenta"), ("baixa", "nominal", "lenta"), ("baixa", "elevado", "moderada"),
          ("média", "leve", "lenta"), ("média", "nominal", "moderada"), ("média", "elevado", "rápida"),
          ("alta", "leve", "moderada"), ("alta", "nominal", "rápida"), ("alta", "elevado", "rápida")]
U = np.round(np.arange(0, 100.0001, 0.5), 6)   # universo discreto da saída, passo 0,5 %

def mu(var, nome, x): return trapezio(x, *VARS[var]["conj"][nome])

## 2. Funções de pertinência

$\mu(x)$ é o **grau** em que $x$ pertence a um conjunto linguístico — não é probabilidade. As funções se sobrepõem: 35 °C é ao mesmo tempo “baixa” (0,5) e “média” (0,33).

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
for a, v in zip(ax, "TLV"):
    lo, hi = VARS[v]["dom"]; xs = np.linspace(lo, hi, 501)
    for nome in VARS[v]["conj"]:
        a.plot(xs, [mu(v, nome, x) for x in xs], lw=2.2, label=nome)
    a.set(title=f"{VARS[v]['nome']} {v}", xlabel=f"{v} ({VARS[v]['unidade']})", ylabel="grau de pertinência μ", ylim=(-0.05, 1.1))
    a.legend()
plt.tight_layout(); plt.show()

## 3. Inferência de Mamdani: regras, agregação e centroide

$$\alpha_k=\min(\mu_{T}(T),\mu_{L}(L)),\qquad \mu(v)=\max_k \min(\alpha_k,\mu_{V_k}(v)),\qquad V^*=\frac{\sum_v v\,\mu(v)}{\sum_v \mu(v)}$$

Se $\sum\mu(v)=0$, nenhuma regra foi acionada e a saída é indefinida (tratada explicitamente).

In [ ]:
def inferir(T, L, regras=REGRAS):
    T, L = min(70, max(20, T)), min(100, max(0, L))
    forcas = [min(mu("T", t, T), mu("L", l, L)) for t, l, _ in regras]
    agreg = np.array([max([min(f, mu("V", v, u)) for f, (_, _, v) in zip(forcas, regras) if f > 0] or [0.0]) for u in U])
    den = agreg.sum()
    return {"forcas": forcas, "agregada": agreg, "V": float((U * agreg).sum() / den) if den > 0 else None}

# cálculo manual: T = 45 °C, L = 70 %
T0, L0 = 45, 70
print("μ_T:", {n: round(mu("T", n, T0), 3) for n in VARS["T"]["conj"]})
print("μ_L:", {n: round(mu("L", n, L0), 3) for n in VARS["L"]["conj"]})
r = inferir(T0, L0)
for k, ((t, l, v), f) in enumerate(zip(REGRAS, r["forcas"]), 1):
    print(f"R{k}: SE T é {t:6s} E L é {l:8s} ENTÃO V é {v:9s} → força {f:.2f}" + ("" if f else "  (não acionada)"))
print(f"V* = {r['V']:.2f} %")
assert abs(r["forcas"][4] - 0.2) < 1e-12 and abs(r["forcas"][5] - 0.6) < 1e-12 and round(r["V"], 1) == 73.6

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].bar([f"R{k}" for k in range(1, 10)], r["forcas"], color=["#146e83" if f else "#c9d4dc" for f in r["forcas"]])
ax[0].set(title=f"Força de cada regra (T = {T0} °C, L = {L0} %)", xlabel="regra", ylabel="α = mín(μ_T, μ_L)", ylim=(0, 1))
for nome in VARS["V"]["conj"]:
    ax[1].plot(U, [mu("V", nome, u) for u in U], lw=1.2, ls="--", label=f"conjunto {nome}")
ax[1].fill_between(U, r["agregada"], color="#7c3aed", alpha=0.3, label="agregação (máx dos cortes)")
ax[1].axvline(r["V"], color="#7c3aed", lw=2.5, label=f"centroide V* = {r['V']:.1f} %")
ax[1].set(title="Agregação dos consequentes e defuzzificação", xlabel="V (%)", ylabel="μ(v)"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 4. Superfície de controle

A superfície $V^*(T, L)$ resume o controlador inteiro. A grade usa a mesma função de inferência; o arquivo `fuzzy_notebook_resultado.json` é gravado para que os testes automáticos comparem com o simulador do navegador.

In [ ]:
Ts, Ls = np.linspace(20, 70, 26), np.linspace(0, 100, 26)
Z = np.array([[inferir(T, L)["V"] for L in Ls] for T in Ts])
json.dump({"T": Ts.tolist(), "L": Ls.tolist(), "V": Z.tolist()}, open("fuzzy_notebook_resultado.json", "w"))
fig = plt.figure(figsize=(15, 5))
a3 = fig.add_subplot(1, 2, 1, projection="3d")
TT, LL = np.meshgrid(Ts, Ls, indexing="ij")
a3.plot_surface(TT, LL, Z, cmap="viridis", edgecolor="k", linewidth=0.2)
a3.set(xlabel="T (°C)", ylabel="L (%)", zlabel="V* (%)", title="Superfície de controle V*(T, L)")
a2 = fig.add_subplot(1, 2, 2)
cs = a2.contourf(Ls, Ts, Z, levels=15, cmap="viridis"); a2.contour(Ls, Ts, Z, levels=[20, 40, 60, 80], colors="w", linewidths=1)
fig.colorbar(cs, ax=a2, label="V* (%)")
a2.set(xlabel="L (%)", ylabel="T (°C)", title="Mapa de calor com curvas de nível")
plt.tight_layout(); plt.show()

## 5. Validação: extremos, regras não acionadas, continuidade e vales locais

In [ ]:
print("Extremo frio  (20 °C, 0 %):", round(inferir(20, 0)["V"], 2), "% — só R1 acionada")
print("Extremo quente (70 °C, 100 %):", round(inferir(70, 100)["V"], 2), "% — só R9; o centroide não chega a 100 %")
print("Base incompleta (só R9) em (25 °C, 10 %):", inferir(25, 10, regras=[REGRAS[8]])["V"], "→ nenhuma regra acionada")
# cobertura: alguma regra sempre acionada no domínio
assert all(inferir(T, L)["V"] is not None for T in np.arange(20, 70.1, 1) for L in np.arange(0, 100.1, 2))

def maior_salto(n):
    Tg, Lg = np.linspace(20, 70, n), np.linspace(0, 100, n)
    Zg = np.array([[inferir(T, L)["V"] for L in Lg] for T in Tg])
    return max(np.abs(np.diff(Zg, axis=0)).max(), np.abs(np.diff(Zg, axis=1)).max())
s26, s101 = maior_salto(26), maior_salto(101)
print(f"Maior salto entre vizinhos: grade 26×26 = {s26:.2f} p.p.; grade 101×101 = {s101:.2f} p.p. (diminui → contínua)")

Lf = np.arange(0, 100.1, 1)
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for T in (25, 32, 45, 60):
    ax[0].plot(Lf, [inferir(T, L)["V"] for L in Lf], label=f"T = {T} °C")
ax[0].set(title="Cortes da superfície: V* em função de L", xlabel="L (%)", ylabel="V* (%)"); ax[0].legend()
vale = [inferir(32, L)["V"] for L in Lf]
ax[1].plot(Lf[20:60], vale[20:60], marker="o", color="#b45309")
ax[1].set(title="Recuo a 32 °C: mais carga, menos ventilação", xlabel="L (%)", ylabel="V* (%)")
plt.tight_layout(); plt.show()
# maior recuo acumulado ao aumentar L, em toda a faixa de T
pior = max(((max(v[:k + 1]) - v[k], T) for T in np.arange(20, 70.1, 0.5)
            for v in [[inferir(T, L)["V"] for L in np.arange(0, 100.1, 0.5)]] for k in range(201)), key=lambda q: q[0])
print(f"V*(32, 35) = {inferir(32, 35)['V']:.2f} %  >  V*(32, 45) = {inferir(32, 45)['V']:.2f} %;  maior recuo em L: {pior[0]:.2f} p.p. (T = {pior[1]} °C)")

**Interpretação — um achado da validação.** A superfície é contínua, mas **não é monótona**: a 32 °C, aumentar o carregamento de 35 % para 45 % reduz o comando de 26,8 % para 21,5 % (−5,3 pontos percentuais). Entre esses valores, R2 (baixa E nominal → lenta) passa de 0,4 para 0,8, e o conjunto “lenta” passa a dominar o centroide; a contribuição “moderada” de R5 fica limitada por μ_média(32 °C) ≈ 0,13. Não é erro de cálculo: é consequência da combinação máximo–mínimo–centroide com esta base de regras, e é fisicamente indesejável. Correções possíveis: rever conjuntos e consequentes (por exemplo, baixa E nominal → moderada), usar outra defuzzificação ou um modelo de Sugeno — discuta o custo de cada opção.

## 6. Comparação com um controle por limiar

In [ ]:
Tf = np.arange(20, 70.1, 0.5)
fig, ax = plt.subplots(figsize=(9, 4))
for L in (20, 50, 80):
    ax.plot(Tf, [inferir(T, L)["V"] for T in Tf], label=f"fuzzy, L = {L} %")
ax.step(Tf, [100 if T >= 45 else 0 for T in Tf], where="post", color="k", ls="--", label="limiar fixo em 45 °C")
ax.set(title="Fuzzy × limiar: decisão gradual e dependente da carga", xlabel="T (°C)", ylabel="comando de ventilação (%)")
ax.legend(); plt.tight_layout(); plt.show()

## 7. Exercícios

1. Calcule à mão $V^*$ para $T = 55$ °C e $L = 30$ %: pertinências, forças das regras e conjuntos cortados. Confira com `inferir(55, 30)` e com a página.
2. Troque o consequente de R7 para “rápida”. O que muda na superfície? A mudança é fisicamente razoável?
3. Reduza o passo do universo para 0,1 % e compare $V^*(45, 70)$. Qual o efeito da discretização?
4. Altere a base de regras para eliminar o recuo de 32 °C (por exemplo, R2 → “moderada”) e meça de novo o maior recuo. O que melhora e o que piora?
5. Remova R5 e procure pontos em que nenhuma regra é acionada. Como o controlador deve se comportar nesses pontos?
6. Explique, para um colega, por que pertinência 0,6 não significa “60 % de probabilidade”.

Respostas esperadas e rubrica: [Exercícios e rubricas](https://furriel.github.io/IA/apoio/exercicios.html).

## Experimentos propostos
1. Calcule `inferir(55, 30)` e compare com a sua conta à mão (V* ≈ 45,8 %).
2. Altere uma regra (por exemplo R2 → “moderada”) e meça de novo o maior recuo da seção 5.
3. Mude o passo do universo de saída e compare V*(45, 70).

## Questões de interpretação
1. Por que a pertinência 0,6 não é uma probabilidade de 60 %?
2. Por que V* nunca chega a 0 % nem a 100 %?
3. O que o vale em 32 °C revela sobre a base de regras?

## Conclusão
Registre: o parâmetro alterado (valor antigo → novo), o V* resultante e o efeito na superfície. As respostas esperadas estão em https://furriel.github.io/IA/apoio/gabaritos.html.

## 8. Experimento guiado — mudar uma regra e interpretar o efeito

**Problema:** no controlador original, a 32 °C, aumentar a carga de 35 % para 45 % pode **reduzir** a ventilação. Será que a troca de uma regra corrige esse comportamento?

1. **Preveja:** qual seria o efeito de trocar a saída da regra R2 (*temperatura baixa E carga nominal*) de **lenta** para **moderada**?
2. **Execute:** o código mantém a base original e cria uma cópia com apenas o consequente de R2 alterado. Os demais conjuntos, regras e entradas continuam iguais.
3. **Compare:** observe as curvas antes × depois, o comando de ventilação para 32 °C e 45 % e o maior recuo dentro do intervalo analisado.
4. **Experimente:** mude só `CONSEQUENTE_R2` para `rápida` ou `lenta` e execute esta célula novamente.
5. **Conclua:** o ajuste melhorou o recuo **no intervalo investigado**? Houve mudança indesejada em outro ponto da curva?

O comentário é derivado dos números calculados; não é LLM. **Um recuo local menor não prova que o controlador seja monotônico em todo o domínio**, nem valida um equipamento elétrico real. Registre as evidências no único relatório final do Moodle.


In [ ]:
# ALTERE APENAS ESTE CONSEQUENTE; original R2 = "lenta".
CONSEQUENTE_R2 = "moderada"  # opções: "lenta", "moderada", "rápida"
assert CONSEQUENTE_R2 in VARS["V"]["conj"], "Escolha lenta, moderada ou rápida."
regras_alteradas = REGRAS.copy()
regras_alteradas[1] = (*REGRAS[1][:2], CONSEQUENTE_R2)  # R2; demais 8 regras intactas
T_EXPERIMENTO = 32
cargas = np.arange(20, 61, 1, dtype=float)
curva_antes = np.array([inferir(T_EXPERIMENTO, L)["V"] for L in cargas])
curva_depois = np.array([inferir(T_EXPERIMENTO, L, regras_alteradas)["V"] for L in cargas])
antes_45 = inferir(32, 45)["V"]
depois_45 = inferir(32, 45, regras_alteradas)["V"]
def pior_recuo_local(valores):
    return float(np.max(np.maximum.accumulate(valores) - valores))
recuo_antes = pior_recuo_local(curva_antes)
recuo_depois = pior_recuo_local(curva_depois)
print("ANTES × DEPOIS — Lógica fuzzy (T = 32 °C)")
print(f"R2 original: lenta; R2 alterada: {CONSEQUENTE_R2}")
print(f"Ventilação em L=45 %: antes {antes_45:.2f} %; depois {depois_45:.2f} %; diferença {depois_45-antes_45:+.2f} p.p.")
print(f"Maior recuo entre L=20 e 60 %: antes {recuo_antes:.2f} p.p.; depois {recuo_depois:.2f} p.p.")
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(cargas, curva_antes, label="original (R2 → lenta)")
ax.plot(cargas, curva_depois, label=f"alterada (R2 → {CONSEQUENTE_R2})")
ax.axvline(45, color="gray", ls=":", label="L=45 %")
ax.set(title="Ventilação a 32 °C: antes × depois", xlabel="carregamento L (%)", ylabel="ventilação V* (%)")
ax.legend(); plt.tight_layout(); plt.show()
print("\nINTERPRETAÇÃO AUTOMÁTICA:")
if recuo_depois < recuo_antes - 1e-6:
    print("A alteração REDUZIU o maior recuo neste corte de temperatura.")
elif recuo_depois > recuo_antes + 1e-6:
    print("A alteração AUMENTOU o maior recuo neste corte de temperatura.")
else:
    print("O maior recuo neste corte não mudou de forma relevante.")
if depois_45 > antes_45 + 1e-6:
    print("A ventilação prevista em (32 °C, 45 %) aumentou.")
elif depois_45 < antes_45 - 1e-6:
    print("A ventilação prevista em (32 °C, 45 %) diminuiu.")
else:
    print("A ventilação prevista em (32 °C, 45 %) permaneceu igual.")
print("Conclusão limitada a estes pontos e a esta faixa: avalie a superfície inteira antes de propor uso físico.")
print("Para o relatório único no Moodle: registre regra, curvas, recuo e limitações.")
